In [1]:
import requests
import json
import re
import pandas as pd
import numpy as np

In [2]:
def get_uniprot(accession):
    url = f'https://rest.uniprot.org/uniprotkb/{accession}'
    return requests.get(url)

In [4]:
get_uniprot('P11473')

<Response [200]>

In [5]:
get_uniprot('helloworld')

<Response [400]>

In [6]:
get_uniprot('helloworld').json()

{'url': 'http://rest.uniprot.org/uniprotkb/helloworld',
 'messages': ["The 'accession' value has invalid format. It should be a valid UniProtKB accession"]}

In [9]:
def uniprot_parse_response(response):
    data = response.json()

    acc = data.get('primaryAccession')
    
    organism = data.get('organism', {})

    if isinstance(organism, dict):
        organism_name = organism.get('scientificName', '')
    else:
        organism_name = organism

    return {acc:
           {'organism': organism_name,
           'geneInfo': data.get('genes', []),
           'sequenceInfo': data.get('sequence', {}),
           'type': 'protein'}}

In [10]:
uniprot_parse_response(get_uniprot('P11473'))

{'P11473': {'organism': 'Homo sapiens',
  'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
       'source': 'HGNC',
       'id': 'HGNC:12679'}],
     'value': 'VDR'},
    'synonyms': [{'value': 'NR1I1'}]}],
  'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
   'length': 427,
   'molWeight': 48289,
   'crc64': 'F95F300D042C4CB7',
   'md5': '0D963ACD4A34674368324EE026023597'},
  'type': 'protein'}}

In [11]:
def get_ensembl(ensembl_id):
    url = f'https://rest.ensembl.org/lookup/id/{ensembl_id}'
    
    headers = {"Content-Type": "application/json"}

    return requests.get(url, headers=headers)

In [12]:
get_ensembl('ENSMUSG00000041147')

<Response [200]>

In [13]:
get_ensembl('helloworld')

<Response [400]>

In [14]:
get_ensembl('helloworld').json()

{'error': "ID 'helloworld' not found"}

In [15]:
get_ensembl('ENSMUSG00000041147').json()

{'assembly_name': 'GRCm39',
 'logic_name': 'ensembl_havana_gene_mus_musculus',
 'source': 'ensembl_havana',
 'object_type': 'Gene',
 'version': 11,
 'canonical_transcript': 'ENSMUST00000044620.11',
 'biotype': 'protein_coding',
 'strand': 1,
 'display_name': 'Brca2',
 'end': 150493794,
 'species': 'mus_musculus',
 'start': 150446095,
 'id': 'ENSMUSG00000041147',
 'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
 'db_type': 'core',
 'seq_region_name': '5'}

In [16]:
def ensembl_parse_response(response):
    data = response.json()

    acc = data.get('id')

    to_return = ['object_type', 'species', 'assembly_name', 'biotype', 
        'display_name', 'id', 'db_type', 'description', 
        'source', 'canonical_transcript']

    return {acc: {key: data.get(key, '') for key in to_return}}

In [17]:
ensembl_parse_response(get_ensembl('ENSMUSG00000041147'))

{'ENSMUSG00000041147': {'object_type': 'Gene',
  'species': 'mus_musculus',
  'assembly_name': 'GRCm39',
  'biotype': 'protein_coding',
  'display_name': 'Brca2',
  'id': 'ENSMUSG00000041147',
  'db_type': 'core',
  'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
  'source': 'ensembl_havana',
  'canonical_transcript': 'ENSMUST00000044620.11'}}

In [21]:
def main(id_list):
    res = {}

    uniprot_pattern = re.compile(r'^[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z]0-9{1,2}$')

    for id in id_list:
        if id.startswith('ENS'):
            response = get_ensembl(id)
            if response.status_code == 200:
                res.update(ensembl_parse_response(response))
            else:
                res[id] = f'error: {response.json().get('error', 'not found')}'
        elif uniprot_pattern.match(id):
            response = get_uniprot(id)

            if response.status_code == 200:
                res.update(uniprot_parse_response(response))
            else:
                res[id] = 'error: not found'

        else:
            res[id] = 'error:unknown database'
        
    return res

In [22]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])

{'P11473': {'organism': 'Homo sapiens',
  'geneInfo': [{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
       'source': 'HGNC',
       'id': 'HGNC:12679'}],
     'value': 'VDR'},
    'synonyms': [{'value': 'NR1I1'}]}],
  'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
   'length': 427,
   'molWeight': 48289,
   'crc64': 'F95F300D042C4CB7',
   'md5': '0D963ACD4A34674368324EE026023597'},
  'type': 'protein'},
 'Q91XI3': {'organism': 'Ictidomys tridecemlineatus',
  'geneInfo': [{'geneName': {'value': 'INS'}}],
  'sequenceInfo': {'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHLVE